# gVXR CPU vs GPU benchmark (Google Colab)

This notebook measures the project’s **gVXR projection engine** on CPU and GPU runtimes. Keep the benchmark settings unchanged between runs. Run the notebook once with **Runtime → Change runtime type → Hardware accelerator: None**, save the JSON result, then repeat in a fresh runtime with **GPU** selected. Compare total scan time and projections/second. Output filenames are based on the active OpenGL renderer, not merely on whether Colab assigned a CUDA GPU.

The notebook does not use the C++ web server. It uses `ct_core.py` and gVXR directly, so it isolates the computation relevant to the PDF requirement. `ct_core.py` handles best-effort renderer selection and reports the active OpenGL device. A GPU runtime only counts as a GPU measurement if the reported renderer identifies NVIDIA hardware.


## 1. Select the project source

If `ct_core.py` is already in the current Colab folder, leave `PROJECT_DIR` as `.`. Otherwise upload `ct_core.py` below.


In [ ]:
from pathlib import Path
PROJECT_DIR = Path('.')
if not (PROJECT_DIR / 'ct_core.py').exists():
    from google.colab import files
    uploaded = files.upload()  # upload ct_core.py
    if 'ct_core.py' not in uploaded:
        raise FileNotFoundError('Please upload ct_core.py')


## 2. Install gVXR dependencies

Run once per runtime. These packages do not build or start the C++ HTTP server.


In [ ]:
import subprocess, sys
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq',
                'build-essential','libxcb-res0','libxcb-xinerama0','libxcb-xkb1','libxcb-icccm4','libxcb-image0',
                'libxcb-keysyms1','libxcb-render-util0','libxcb-shape0','libxcb-randr0','libxkbcommon-x11-0',
                'libxrandr2','libxinerama1','libxcursor1','libxi6','libxxf86vm1',
                'libgl1','libegl1','libopengl0','libglu1-mesa','libgl1-mesa-dri','libegl-mesa0','libglx-mesa0',
                'libxcb-composite0','libxcb-cursor0','libxcb-dri2-0','libxcb-ewmh2',
                'libxcb-dri3-0','libxcb-glx0','libxcb-present0','libxcb-sync1','libxcb-xfixes0','libxcb-xinput0',
                'libxcb-util1','libxcb-xv0','libxcb-damage0','libxcb-dpms0','libxcb-record0','libxcb-screensaver0',
                'libxcb-shm0','libxcb-xtest0','libxcb-xf86dri0','libxcb-xvmc0'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','gvxr','numpy','trimesh'], check=True)


## 3. Confirm runtime and renderer

This cell calls gVXR through `ct_core.py`. The core makes the NVIDIA EGL selection before context creation, then reports both the requested selection and the OpenGL renderer actually in use. The OpenGL renderer is the evidence for whether the projections ran on the GPU.


In [ ]:
import os, sys, subprocess, json, time, platform
import numpy as np
import trimesh
sys.path.insert(0, str(PROJECT_DIR.resolve()))
import ct_core as core

def runtime_info():
    info = {'python': sys.version.split()[0], 'platform': platform.platform()}
    try:
        p = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], capture_output=True, text=True, timeout=10)
        info['nvidia_gpu'] = p.stdout.strip() if p.returncode == 0 else 'not detected'
    except Exception:
        info['nvidia_gpu'] = 'not detected'
    try:
        info['opengl'] = core.on_gl_thread(lambda: (core._gx(), core.renderer_info())[1])
        info['renderer_selection'] = core.renderer_selection_info()
        info['egl_vendor_library_filenames'] = os.environ.get('__EGL_VENDOR_LIBRARY_FILENAMES', 'system default')
        info['nvidia_egl_library_found'] = bool(info['renderer_selection'].get('nvidia_egl_library'))
    except Exception as e:
        info['opengl_error'] = repr(e)
    return info

info = runtime_info()
print(json.dumps(info, indent=2))


## 4. Run the fixed benchmark

The default scan is 30 projections at 160 × 80 pixels of a generated 20 mm water-equivalent box. It warms the renderer, then performs three measured scans. Scene setup and projection acquisition are timed separately. Change the constants only if you also use the same values in both runtimes.


In [ ]:
# Fixed workload — use these exact values for CPU and GPU runs.
NX, NY = 160, 80
N_ANGLES = 30
REPEATS = 3
WARMUP_ANGLES = 2

mesh_path = Path('/content/gvxr_benchmark_box.stl')
trimesh.creation.box(extents=(20, 20, 20)).export(mesh_path)
obj = core.ObjectCfg(name='benchmark_box', stl_path=str(mesh_path), kind='Compound', material='H2O', density=1.0)
cfg = core.ScanCfg(objects=[obj], stl_unit='mm', beam='Cone', source_x_cm=40, detector_x_cm=10,
                   energy_kev=80, photons=1000, nx=NX, ny=NY, pixel_mm=1.0,
                   n_angles=N_ANGLES, total_deg=360, noise=False)

def one_scan(n_angles):
    t0 = time.perf_counter()
    core.on_gl_thread(core.build_scene, cfg)
    setup_s = time.perf_counter() - t0
    t1 = time.perf_counter()
    projection_stack = core.on_gl_thread(core.acquire, n_angles, 360.0 / N_ANGLES)
    acquisition_s = time.perf_counter() - t1
    return setup_s, acquisition_s, projection_stack

# Warm-up (excluded from measurements)
_, _, warm = one_scan(WARMUP_ANGLES)
del warm
rows = []
for i in range(REPEATS):
    setup_s, acquisition_s, stack = one_scan(N_ANGLES)
    rows.append({'repeat': i + 1, 'scene_setup_s': setup_s, 'projection_acquisition_s': acquisition_s,
                 'total_s': setup_s + acquisition_s, 'projections_per_s': N_ANGLES / acquisition_s,
                 'shape': list(stack.shape), 'min': float(np.nanmin(stack)), 'max': float(np.nanmax(stack))})
    del stack

acq = [r['projection_acquisition_s'] for r in rows]
tot = [r['total_s'] for r in rows]
result = {'environment': info, 'workload': {'nx': NX, 'ny': NY, 'angles': N_ANGLES, 'repeats': REPEATS,
          'warmup_angles': WARMUP_ANGLES, 'beam': cfg.beam, 'energy_keV': cfg.energy_kev,
          'mesh': '20 mm cube, H2O, 1 g/cm3'}, 'runs': rows,
          'summary': {'median_acquisition_s': float(np.median(acq)),
                      'median_total_s': float(np.median(tot)),
                      'median_projections_per_s': float(N_ANGLES / np.median(acq))}}
print(json.dumps(result, indent=2))


Run this cell after the benchmark. Save the downloaded JSON, then run the notebook in the other runtime and save that JSON too. Compare `summary.median_acquisition_s` and `summary.median_projections_per_s`. The JSON records the OpenGL vendor and renderer for inspection, but does not infer whether rendering was done in software or on a GPU. `nvidia-smi` records GPU availability, not which device performed OpenGL rendering.

In [ ]:
from google.colab import files
out = Path('gvxr_benchmark.json')
out.write_text(json.dumps(result, indent=2))
print('Saved', out)
files.download(str(out))


## Interpretation and limitations

- Compare the **median acquisition time** for the same number of projections and detector pixels. Higher projections/second means faster projection computation.
- `scene_setup_s` includes source/detector setup, mesh loading, and material assignment; `projection_acquisition_s` measures repeated `computeXRayImage()` calls plus rotations.
- This is a controlled small benchmark, not a full clinical workload. Repeat with the project’s representative meshes and larger detector sizes for a stronger result.
- Colab’s CPU and GPU machines can have different CPU models, memory, drivers, and OpenGL implementations. Preserve the JSON files and report these conditions.
- Record whether the Colab runtime was configured as CPU or GPU alongside each benchmark result. The reported OpenGL renderer is useful diagnostic information, but this notebook does not classify it as hardware or software rendering.
